# W5 Lab — Reflection, then Evaluation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week05/W5_lab_reflection_evals.ipynb)

**Goal.** By the end of this lab you can run the chapter's two halves as one
experiment. Part A implements the reflection pattern — draft, critique ✍️, revise —
and backs the claim "V2 beats V1" with a judge score. Part B turns that measuring
habit into the full discipline on a different component: a research step scored
against a fixed evalset, error analysis over the failures, and prompt iteration
against a numeric target ✍️, with the judge itself written ✍️ and checked against an
objective metric.

Why this week: the lecture (notes Ch. 5) argued that reflection's validity stands on
an evaluation signal, and that the claim "it helped" is itself an evaluation. The lab
is that argument in code: first the loop that improves an output, then the machinery
that proves the improvement.

The path: setup → Part A: baseline draft → critique ✍️ → revision → improvement with
a number, plus two prediction exercises → Part B: research tools → the research step →
a component metric → the measured task ✍️ (core) → error analysis → threshold and
judge exercises ✍️ → completion.

> Part A adapted from DeepLearning.AI, *Agentic AI* by Andrew Ng — Module 2
> (graded lab GL-M2). Part B from Module 4 (Evaluations and Error Analysis), with
> the source's Tavily search replaced by keyless arXiv and Wikipedia APIs.

*Runtime:* Google Colab, top-to-bottom, ~85 minutes. Cells marked ✍️ ask for your own writing — a fill-in or a written prediction.


## 1. Setup

Same setup as W1–W2: client library, key, helper, one test call.

### 1.1 Installation

`aisuite` exposes multiple providers behind one interface, so lab code stays identical
whichever provider your key belongs to.

*Do:* run the cell below (about 30 seconds, once per session).

In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

Paste your API key over `PASTE-YOUR-KEY-HERE` (issuing steps: the API Setup guide on
the course site). This lab is text-only; any chat model works.

*Do:* replace `PASTE-YOUR-KEY-HERE` with your key and run the cell.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: "anthropic:claude-haiku-4-5"

### 1.3 Client and helper

`ask` sends a single text prompt and returns the reply. `n_calls` counts API calls,
read in the Section 5 exercises.

*Do:* run the cell unchanged.

In [ ]:
import json
import re

import aisuite

client = aisuite.Client()
n_calls = 0                            # API call counter


def ask(prompt, model=None, temperature=0.7):
    """Prompt string -> assistant reply string."""
    global n_calls
    n_calls += 1
    response = client.chat.completions.create(
        model=model or MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content

### 1.4 Verification

*Do:* run the cell and confirm the output is exactly `ready`.

In [ ]:
print(ask("Reply with exactly: ready"))

If the output is `ready`, key and billing are correct. Any error at this point is a setup problem, not a code problem.

## Part A — One Output, Fed Back

Sections 2–5 run reflection on a single essay: draft, critique, revise, and one
measured comparison at the end.

## 2. One Call, No Checking — the Baseline

The question this section answers: what does a generation call produce when nothing
reads its output? The essay prompt below is the one used in the source lab.
`generate_draft` is given in full — the drafting step is ordinary prompting; the new
machinery of this lab starts in Section 3.

*Do:* run the cell and read the draft as a grader would.

In [ ]:
ESSAY_PROMPT = "Should social media platforms be regulated by the government?"


def generate_draft(prompt, model=None):
    """Essay prompt -> first draft (V1). Plain generation, no checking."""
    instruction = (
        "Write a well-structured draft essay of about 250 words in response to "
        "the following prompt. The draft should include an introduction, body, "
        "and conclusion.\n\nPrompt:\n" + prompt
    )
    return ask(instruction, model=model)


draft_v1 = generate_draft(ESSAY_PROMPT)
print(draft_v1)

A first draft is rarely wrong, but it is rarely strong: claims asserted without
evidence, no engagement with the opposing view, a conclusion that restates the
introduction. In one-shot prompting these defects ship, because the workflow contains
no step that reads the output. Reflection adds that step — and it works because of an
asymmetry the theory chapter develops (notes §5.1): pointing at a defect in a finished
text is an easier task than producing the text, so a model can find faults in its own
output that it did not avoid while writing.

## 3. The Critique ✍️

The reflection step is one `ask` call whose prompt tells the model to criticize, not
to rewrite. What concrete criticism looks like, on a two-sentence example:

In [ ]:
EXAMPLE_TEXT = ("Social media is bad for democracy. Therefore the government "
                "should regulate it.")

EXAMPLE_REFLECTION_PROMPT = (
    "You are a strict writing reviewer. Criticize the following text. Do not "
    "rewrite it. Name each concrete defect in argument, evidence, structure, or "
    "clarity, and for each defect state what a fix would need to add.\n\nText:\n"
)

print(ask(EXAMPLE_REFLECTION_PROMPT + EXAMPLE_TEXT))

The example output names defects ("asserts harm without evidence", "no opposing
view considered") and states what a fix needs — it does not produce the fixed text.
That separation is the whole design: the critique's only job is to give the revision
step something concrete to act on.

The fill-in: `REFLECTION_PROMPT`, the critique instruction for essay drafts. The
draft is appended after it by `reflect_on_draft`, which is given.

Requirements for the prompt — it must instruct the model to:

- criticize only, not rewrite;
- cover structure, clarity, strength of argument, and evidence;
- name each defect concretely enough that a revision can act on it.

*Do:* the starter prompt between the markers is too vague to satisfy any of the
three requirements — rewrite it so it satisfies all of them, then run the cell.

In [ ]:
### FILL IN (START) ###
REFLECTION_PROMPT = """\
Give feedback on the essay draft below.
"""
### FILL IN (END) ###


def reflect_on_draft(draft, model=None):
    """Draft essay -> critique text (no rewrite)."""
    return ask(REFLECTION_PROMPT + "\n\nDraft:\n" + draft, model=model)

In [ ]:
feedback = reflect_on_draft(draft_v1)
print(feedback)

Read the feedback against the requirements: every point should name a defect and
what fixing it requires; no sentence should be rewritten essay text. If the model
rewrote instead of criticizing, the prompt is missing the criticize-only instruction —
sharpen it and rerun.

## 4. The Revision

The final step feeds both the original draft and the feedback to a third call, which
returns only the improved essay. `revise_draft` is given — like drafting, it is
ordinary prompting; the critique in the middle is what makes the three calls a
reflection loop.

*Do:* run the cell and compare V2 against the feedback — each defect the critique
named should be absent from V2.

In [ ]:
def revise_draft(original_draft, reflection, model=None):
    """(Draft, critique) -> revised essay (V2). Returns only the essay."""
    instruction = (
        "Revise the following essay draft using the feedback provided. Address "
        "every point in the feedback. Keep it about 250 words. Return only the "
        "improved essay.\n\nOriginal Draft:\n" + original_draft
        + "\n\nFeedback:\n" + reflection
    )
    return ask(instruction, model=model)


draft_v2 = revise_draft(draft_v1, feedback)
print(draft_v2)

## 5. Improvement with a Number ✍️ (core)

The claim "V2 is better than V1" becomes a measurement. **LLM-as-judge** = using a
model call to grade an output against a stated criterion (the judge is studied in Part B
of this lab; here it is only used). The judge reads an
essay and awards one point per criterion:

| Criterion | Awarded when the essay... |
|---|---|
| `thesis` | states a clear position on the question |
| `structure` | has a recognizable introduction, body, and conclusion |
| `evidence` | supports claims with concrete examples or reasons |
| `counterargument` | addresses at least one opposing view |

Target: V2 total ≥ 3, and V2 > V1. If the target is missed, sharpen
`REFLECTION_PROMPT` and rerun Sections 3–4 — the critique prompt is the tunable part
of this pipeline.

*Do:* run the collapsed judge cell as-is; on a miss, sharpen `REFLECTION_PROMPT` and
rerun Sections 3–4.

In [ ]:
#@title Judge code — run as-is (applies the criteria table above) { display-mode: "form" }
CRITERIA = {
    "thesis": "states a clear position on the question",
    "structure": "has a recognizable introduction, body, and conclusion",
    "evidence": "supports claims with concrete examples or reasons",
    "counterargument": "addresses at least one opposing view",
}

TARGET_SCORE = 3


def judge_essay(essay, question):
    """(Essay text, essay question) -> {criterion: 0/1} dict."""
    criteria_text = "\n".join(f'- "{name}": 1 if the essay {desc}, else 0'
                              for name, desc in CRITERIA.items())
    prompt = (
        "You grade an essay against its question.\n"
        "Return a JSON object only, no prose, with these integer fields:\n"
        + criteria_text
        + "\n\nQuestion:\n" + question
        + "\n\nEssay:\n" + essay
    )
    reply = ask(prompt, temperature=0.0)
    found = re.search(r"\{[\s\S]*\}", reply)
    scores = json.loads(found.group(0)) if found else {}
    return {name: int(scores.get(name, 0)) for name in CRITERIA}


score_v1 = judge_essay(draft_v1, ESSAY_PROMPT)
score_v2 = judge_essay(draft_v2, ESSAY_PROMPT)
total_v1, total_v2 = sum(score_v1.values()), sum(score_v2.values())

print("V1:", score_v1, "->", total_v1)
print("V2:", score_v2, "->", total_v2)
print("target met:", total_v2 >= TARGET_SCORE and total_v2 > total_v1)

The per-criterion dict shows where the gain came from — typically `evidence` and
`counterargument`, the defects a critique names most reliably. A gain that appears
here without a corresponding point in the feedback text means the revision model fixed
it on its own; the loop still worked, but the critique did not cause that part.

### Exercise — a second reflection round ✍️

Whether a second round of critique-and-revise keeps paying: reflect on V2, revise into
V3, judge V3. A round costs three more calls. Prediction, written down first: does V3
improve on V2 by enough to justify them, or has the score saturated?

*Do:* run the cell and compare with your prediction.

In [ ]:
calls_before = n_calls

feedback_2 = reflect_on_draft(draft_v2)
draft_v3 = revise_draft(draft_v2, feedback_2)
score_v3 = judge_essay(draft_v3, ESSAY_PROMPT)

print("V2:", total_v2, " V3:", sum(score_v3.values()),
      " extra calls:", n_calls - calls_before)

On a 4-point checklist the second round usually adds little: the first critique
already consumed the defects the judge can see, and each round costs three more calls.
Reflection converges when no new information enters the loop — the homework notebook
breaks exactly this limit by grounding the critique in a rendered chart image, and W4
breaks it with tools that bring external facts in.

### Exercise — critique without criteria ✍️

What the requirements in Section 3 buy: the cell runs the same loop once more with a
vague critique instruction in place of yours. Prediction, written down first: which
V2 scores higher, and on which criteria?

*Do:* run the cell and compare with your prediction.

In [ ]:
VAGUE_PROMPT = "Say how to improve this essay."

feedback_vague = ask(VAGUE_PROMPT + "\n\nDraft:\n" + draft_v1)
draft_v2_vague = revise_draft(draft_v1, feedback_vague)
score_v2_vague = judge_essay(draft_v2_vague, ESSAY_PROMPT)

print("V2 (your prompt):", total_v2,
      " V2 (vague prompt):", sum(score_v2_vague.values()))

The vague critique tends to produce generic advice ("make it more engaging"), and
the revision improves less or drifts. The criteria in the critique prompt are doing
the same work a rubric does for a human reviewer: they turn "improve" into checkable
obligations.

## Part B — From One Comparison to a Discipline

Part A produced one number on one essay: V2 beat V1 under a four-criterion judge.
That is a comparison, not yet an evaluation — the second half fixes the dataset, the
metric, and the procedure (notes Ch. 5 §5.7), on a fresh component: a research step
that gathers sources for a topic. The component changes; the measuring habit is the
point.


## 6. Research Tools

The research step needs external information sources. Two keyless tools serve it: an
arXiv search (public API, XML response) and a Wikipedia summary lookup (public API,
JSON response). Both return a list of dicts and return an error dict instead of
raising, so a failed lookup reaches the model as text rather than crashing the run.
*Do:* run the cell unchanged.


In [ ]:
import requests
import xml.etree.ElementTree as ET

HTTP_TIMEOUT = 20   # seconds; both APIs normally answer well under this

session = requests.Session()
session.headers.update({"User-Agent": "STML-2026-lab/1.0"})

def arxiv_search_tool(query: str, max_results: int = 5) -> list:
    """
    Searches arXiv for research papers matching the given query.
    Returns a list of dicts with title, authors, published date, url, and summary.
    """
    url = (f"https://export.arxiv.org/api/query?search_query=all:{query}"
           f"&start=0&max_results={max_results}")
    try:
        response = session.get(url, timeout=HTTP_TIMEOUT)
        response.raise_for_status()
        root = ET.fromstring(response.content)
        ns = {"atom": "http://www.w3.org/2005/Atom"}
        results = []
        for entry in root.findall("atom:entry", ns):
            results.append({
                "title": entry.find("atom:title", ns).text.strip(),
                "authors": [a.find("atom:name", ns).text
                            for a in entry.findall("atom:author", ns)],
                "published": entry.find("atom:published", ns).text[:10],
                "url": entry.find("atom:id", ns).text,
                "summary": entry.find("atom:summary", ns).text.strip()[:300],
            })
        return results
    except Exception as exc:
        return [{"error": str(exc)}]

def wikipedia_search_tool(query: str, sentences: int = 5) -> list:
    """
    Searches Wikipedia and returns a summary of the best-matching article.
    Returns a list with one dict containing title, summary, and url.
    """
    api = "https://en.wikipedia.org/w/api.php"
    try:
        found = session.get(api, params={
            "action": "opensearch", "search": query, "limit": 1,
            "format": "json"}, timeout=HTTP_TIMEOUT).json()
        if not found[1]:
            return [{"error": f"no Wikipedia article found for {query!r}"}]
        title = found[1][0]
        page = session.get(api, params={
            "action": "query", "prop": "extracts", "exintro": 1,
            "explaintext": 1, "titles": title, "format": "json"},
            timeout=HTTP_TIMEOUT).json()
        extract = next(iter(page["query"]["pages"].values())).get("extract", "")
        return [{
            "title": title,
            "summary": " ".join(extract.split(". ")[:sentences]),
            "url": "https://en.wikipedia.org/wiki/" + title.replace(" ", "_"),
        }]
    except Exception as exc:
        return [{"error": str(exc)}]

A direct call shows the shape of what the model will receive.
*Do:* run the cell and skim one returned dict.


In [ ]:
for paper in arxiv_search_tool("large language model agents", max_results=2):
    print(paper.get("title", paper), "\n  ", paper.get("url", ""))

## 7. Research Step — `find_references`

The research function hands the task and the tools to the model; `aisuite` runs the
tool calls (up to `max_turns` rounds) and returns the final text (notes Ch. 3: function
calling). The prompt template is a parameter, because the measured task in Section 9
replaces it.

In [ ]:
from datetime import date

BASELINE_RESEARCH_PROMPT = """\
You are a research function with access to:
- arxiv_search_tool: academic papers
- wikipedia_search_tool: encyclopedic summaries

Task:
{task}

Today is {today}."""

def find_references(task, prompt_template=BASELINE_RESEARCH_PROMPT):
    """Research task -> model's final text after tool use."""
    prompt = prompt_template.format(task=task, today=date.today().isoformat())
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            tools=[arxiv_search_tool, wikipedia_search_tool],
            max_turns=5,
        )
        return response.choices[0].message.content
    except Exception as exc:
        return f"[Model Error: {exc}]"

The first run uses the source lab's example task.
*Do:* run the cell; the research output prints below it.


In [ ]:
research_task = "Find 2 recent papers about recent developments in black hole science"
research_result = find_references(research_task)
print(research_result)

Read the output once as a user would: the summaries look plausible. Whether the
sources are trustworthy, and whether they are even linked, is not visible from
plausibility — that judgment needs a check that code can repeat.

## 8. Component-Level Evaluation — Preferred Domains

**Component-level evaluation** = evaluating one step of a workflow in isolation
instead of rerunning the whole pipeline (notes §5.8). Rerunning search → draft →
reflect end-to-end for every search tweak is expensive, and later components add
randomness that hides small search improvements; evaluating the search step alone
gives a clearer signal at lower cost.

The check here is objective with a per-example ground truth: extract every URL from
the research output, compare each against a predefined set of preferred domains, and
compute the preferred-to-total ratio. PASS if the ratio clears a threshold.

In [ ]:
import re

PREFERRED_DOMAINS = {
    # General reference / institutions / publishers
    "wikipedia.org", "nature.com", "science.org", "cell.com",
    "mit.edu", "stanford.edu", "harvard.edu", "nasa.gov", "noaa.gov",
    # CS/AI venues & indexes
    "arxiv.org", "acm.org", "ieee.org", "neurips.cc", "icml.cc", "openreview.net",
    # Other reputable outlets
    "elifesciences.org", "pnas.org", "jmlr.org", "springer.com", "sciencedirect.com",
}

URL_PATTERN = re.compile(r"https?://[^\s\]\)>\}]+", flags=re.IGNORECASE)

def evaluate_reference_domains(top_domains, raw, min_ratio=0.4):
    """(preferred domain set, output text, threshold) -> (pass flag, report text)."""
    urls = URL_PATTERN.findall(raw)
    if not urls:
        return False, "No URLs detected in the provided text."

    details, preferred_count = [], 0
    for url in urls:
        domain = url.split("/")[2]
        preferred = any(td in domain for td in top_domains)
        preferred_count += preferred
        details.append(f"- {url} -> {'PREFERRED' if preferred else 'NOT preferred'}")

    ratio = preferred_count / len(urls)
    flag = ratio >= min_ratio
    report = (f"total: {len(urls)}  preferred: {preferred_count}  "
              f"ratio: {ratio:.0%}  threshold: {min_ratio:.0%}  "
              f"status: {'PASS' if flag else 'FAIL'}\n" + "\n".join(details))
    return flag, report

The evaluation applied to the Section 7 output:

In [ ]:
flag, report = evaluate_reference_domains(PREFERRED_DOMAINS, research_result)
print(report)

The check is reproducible and costs no model call: the same text always yields the
same ratio, because the ground truth (preferred or not, per URL) is defined
explicitly. A FAIL here localizes the problem to the research step — no drafting or
reflection component needs to be rerun or inspected.

## 9. Measured Task — Prompt Improvement Against an Evalset ✍️ (core)

One example is not a measurement. Following the source lab's own extension, the
evaluation now runs over a small evalset of research topics, each scored by the
preferred-domain ratio.

| Constant | Value |
|---|---|
| evalset | 5 topics |
| per-topic threshold | ratio ≥ 0.4 |
| **target** | **≥ 4 of 5 topics PASS** |

The lever is the research prompt. The baseline template above says nothing about
citing URLs or preferring academic sources, and its score shows it.

In [ ]:
EVALSET_TOPICS = [
    "evaluation of LLM agents",
    "CRISPR gene editing",
    "exoplanet atmosphere observations",
    "reinforcement learning from human feedback",
    "quantum error correction",
]

MIN_RATIO = 0.4
TARGET_PASSES = 4

def run_evalset(prompt_template):
    """Prompt template -> (pass count, list of output texts); prints one row per topic."""
    passes, outputs = 0, []
    for topic in EVALSET_TOPICS:
        task = f"Find 2-3 key papers and reliable overviews about {topic}."
        output = find_references(task, prompt_template)
        outputs.append(output)
        flag, report = evaluate_reference_domains(
            PREFERRED_DOMAINS, output, min_ratio=MIN_RATIO)
        ratio_line = report.splitlines()[0]
        passes += flag
        print(f"{'PASS' if flag else 'FAIL'}  {topic:45s} {ratio_line}")
    print(f"passed {passes}/{len(EVALSET_TOPICS)}")
    return passes, outputs

### 9.1 Baseline measurement
*Do:* run the cell; one PASS/FAIL row prints per topic.


In [ ]:
baseline_passes, baseline_outputs = run_evalset(BASELINE_RESEARCH_PROMPT)

Typical baseline failures, visible in the per-URL details: outputs with no URLs at
all (the prompt never asked for links), and sources drawn from blogs or news
aggregators rather than the venues in the preferred set. Both are prompt problems,
not tool problems — the arXiv tool was available and underused.

### 9.2 Research prompt ✍️

Write `RESEARCH_PROMPT`. Requirements: it must keep the placeholders `{task}` and
`{today}`; it must name the two available tools; and it must state what the
evaluation checks — every source listed with its full URL, academic and institutional
sources preferred, papers found through the arXiv tool. Do not copy the baseline
header verbatim; state the source policy in your own words.

In [ ]:
### FILL IN (START) ###
RESEARCH_PROMPT = (
    "{task}\n\nToday is {today}."
)
### FILL IN (END) ###

### 9.3 Measurement

Run, read which topics fail and why (the per-URL details name every miss), adjust one
instruction, run again. The target is reached when at least 4 of 5 topics pass.

In [ ]:
improved_passes, improved_outputs = run_evalset(RESEARCH_PROMPT)
print(f"baseline {baseline_passes}/5  ->  improved {improved_passes}/5")
print("TARGET REACHED" if improved_passes >= TARGET_PASSES else "KEEP ITERATING")

## 10. Error Analysis

**Error analysis** = reading the failing examples before changing anything, so the
next change addresses an observed failure rather than a guess. The cell prints the
full per-URL detail for every topic that still fails; each NOT-preferred line is a
concrete candidate: either the prompt should steer away from that source, or the
domain genuinely belongs in `PREFERRED_DOMAINS` and the ground truth is what needs
editing.
*Do:* run the cell and read every failing row before touching the prompt.


In [ ]:
for topic, output in zip(EVALSET_TOPICS, improved_outputs):
    flag, report = evaluate_reference_domains(
        PREFERRED_DOMAINS, output, min_ratio=MIN_RATIO)
    if not flag:
        print(f"== {topic}\n{report}\n")
print("(no failing topics)" if improved_passes == len(EVALSET_TOPICS) else "")

## 11. Exercises ✍️

Protocol: write your prediction down first, run the cell, compare.

### 11.1 Threshold strictness

The stored outputs are re-scored at four thresholds — no new model calls. Prediction:
at which threshold does the pass count first drop below the target?

In [ ]:
for threshold in (0.2, 0.4, 0.6, 0.8):
    passes = sum(
        evaluate_reference_domains(PREFERRED_DOMAINS, output, min_ratio=threshold)[0]
        for output in improved_outputs)
    print(f"threshold {threshold:.0%}: {passes}/{len(improved_outputs)} pass")

A stricter threshold makes the eval harder to satisfy without making the system
better; a looser one passes outputs the check was written to catch. The threshold is
part of the eval's design, and changing it mid-project silently redefines what
"improving" means.

### 11.2 LLM judge vs. objective check ✍️

The domain ratio is objective; source *quality* judgments beyond the domain list are
not. **LLM-as-judge** = using a model call to grade an output against stated criteria
(notes Ch. 5 §5.9). Write `JUDGE_PROMPT`: it must keep the placeholder `{text}`, state the
criteria (sources linked, reputable, recent, relevant), and demand a single integer
score 1–5 on the last line as `SCORE: <n>`.

Prediction: will the judge's per-topic ranking agree with the objective ratio's
ranking?

In [ ]:
### FILL IN (START) ###
JUDGE_PROMPT = (
    "{text}"
)
### FILL IN (END) ###

def judge_score(text):
    """Research output -> judge score 1-5 (0 if unparseable)."""
    reply = ask(JUDGE_PROMPT.format(text=text[:4000]))
    match = re.search(r"SCORE:\s*([1-5])", reply)
    return int(match.group(1)) if match else 0

def domain_ratio(text):
    """Research output -> preferred-domain ratio."""
    urls = URL_PATTERN.findall(text)
    if not urls:
        return 0.0
    return sum(any(td in u.split("/")[2] for td in PREFERRED_DOMAINS)
               for u in urls) / len(urls)

print(f"{'topic':45s} {'ratio':>6s} {'judge':>6s}")
for topic, output in zip(EVALSET_TOPICS, improved_outputs):
    print(f"{topic:45s} {domain_ratio(output):6.0%} {judge_score(output):6d}")

Where the two columns disagree, the disagreement itself is information: the judge
sees qualities the domain list cannot (relevance, recency) and misses ones it can
(a made-up URL on a reputable domain still counts as text to the judge). The
objective check is the anchor; the judge extends it where no ground truth exists —
at the price of one model call per example and its own variance.

## 12. Completion Check

All rows must read PASS before submission; grading checks these same structural
facts, never prose quality.

In [ ]:
completion = {
    "Part A: REFLECTION_PROMPT strengthened beyond the starter":
        len(REFLECTION_PROMPT.strip()) > 60
        and "Give feedback on the essay draft" not in REFLECTION_PROMPT,
    "Part A: workflow produced a revision":
        isinstance(draft_v2, str) and len(draft_v2) > 0 and draft_v2 != draft_v1,
    "Part A: judge returned all criteria":
        set(score_v2) == set(CRITERIA),
    "Part A: target met (V2 >= 3 and V2 > V1)":
        total_v2 >= TARGET_SCORE and total_v2 > total_v1,
    "RESEARCH_PROMPT keeps {task} and {today}":
        "{task}" in RESEARCH_PROMPT and "{today}" in RESEARCH_PROMPT,
    "RESEARCH_PROMPT written (>= 80 chars)": len(RESEARCH_PROMPT.strip()) >= 80,
    "JUDGE_PROMPT keeps {text}":             "{text}" in JUDGE_PROMPT,
    "JUDGE_PROMPT written (>= 60 chars)":    len(JUDGE_PROMPT.strip()) >= 60,
    "evalset measured (5 outputs)":          len(improved_outputs) == 5,
    "target reached (>= 4/5 pass)":          improved_passes >= TARGET_PASSES,
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")

---

**Homework:** `W5_hw_chart_reflection.ipynb` — the same generate → reflect → revise
loop where the output is a rendered matplotlib chart and the critique reads the image
itself: an external signal in exactly the sense of notes §5.5. Due before W6.

W6 divides one agent's work among several: a multi-agent pipeline where specialized
roles hand results to each other, from *Agentic AI* Module 5. Reference answers for this
lab and the homework: `labs/checkpoints/week05/solution.py`, published after the
homework deadline.
